<img src="assets/banner.png" alt="Banner" style="width:100%; border-radius:10px; margin-bottom:20px;">

# 🚀 01 - Natural Language Processing: Model Training
**Author:** Mostafa  
**License:** MIT License

Hello! I built this notebook to orchestrate the end-to-end training pipeline for our NLP model. My goal was to take raw consumer complaints and train a highly robust AI to classify them correctly.

I decided to fine-tune `roberta-base`. To make the training incredibly fast and efficient on dual GPUs, I implemented Mixed Precision (FP16). I also added Early Stopping to prevent overfitting and utilized dynamic class weights so the model doesn't ignore rare complaint categories. Let's dive in!

In [ ]:
# --- 1. Import Essential Libraries & Setup Environment ---
import os
import json
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

# Lock random seeds for reproducible results across different runs
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# --- 2. Configuration & Path Resolution ---
import glob
import os

print("--- Scanning Kaggle Input Directory ---")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
print("-" * 40 + "\n")

# Dynamically locate the dataset based on potential Kaggle directory names
exact_paths = [
    "/kaggle/input/consumer-complaints-dataset-for-nlp/complaints_processed.csv",
    "/kaggle/input/consume-complaints-dataset-fo-nlp/complaints_processed.csv"
]
DATA_PATH = None
for p in exact_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break

# Fallback: Find the first CSV file available
if DATA_PATH is None:
    csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not csv_files:
        raise FileNotFoundError("Could not find ANY .csv dataset in /kaggle/input/. Please check the printed files above.")
    DATA_PATH = csv_files[0]
    
print(f"✅ Successfully located dataset at: {DATA_PATH}\n")

# --- HYPERPARAMETERS ---
MODEL_NAME = "roberta-base"    # Core NLP Architecture
MAX_LEN = 256                  # Truncate/Pad length (optimized for speed vs context)
BATCH_SIZE = 64                # Distributed across available GPUs
EPOCHS = 5                     # Maximum training cycles (regulated by Early Stopping)
LEARNING_RATE = 2e-5           # Peak learning rate for AdamW

# Setup Local Output Directories
CURRENT_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(CURRENT_DIR, "outputs")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# --- 3. Data Ingestion & Cleaning ---
# Load the dataset into a Pandas DataFrame
df = pd.read_csv(DATA_PATH)

# Clean null or purely whitespace entries from the narrative column
df['narrative'] = df['narrative'].fillna("")
df = df[df['narrative'].str.strip() != ""]

print(f"📊 Total valid complaints ready for processing: {len(df)}")

In [ ]:
# --- 4. Tokenization & Dataset Preparation ---
# Map string categories to numerical indices for the neural network
labels = df['product'].unique()
label2idx = {label: i for i, label in enumerate(labels)}
idx2label = {i: label for label, i in label2idx.items()}

# Export the mapping for future inference sessions
with open(os.path.join(CHECKPOINT_DIR, "label_mappings.json"), "w") as f:
    json.dump({"label2idx": label2idx, "idx2label": idx2label}, f)

# Stratified Split: Ensure equal class representation in both Train and Test sets
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['product'])

# Compute Class Weights to balance out rare complaints
class_weights_arr = compute_class_weight('balanced', classes=np.unique(train_df['product']), y=train_df['product'])
weights_ordered = [class_weights_arr[np.where(np.unique(train_df['product']) == idx2label[i])[0][0]] for i in range(len(labels))]

# Initialize the RoBERTa Fast Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.save_pretrained(CHECKPOINT_DIR)

# Custom PyTorch Dataset to handle text encoding on the fly
class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        # Convert text into machine-readable numerical tokens
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Instantiate Datasets and DataLoaders for batching
train_dataset = ComplaintDataset(train_df['narrative'].values, [label2idx[l] for l in train_df['product'].values], tokenizer, MAX_LEN)
test_dataset = ComplaintDataset(test_df['narrative'].values, [label2idx[l] for l in test_df['product'].values], tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2)

In [ ]:
# --- 5. Model Architecture & Optimizer Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Computation Device: {device}")

# Load the base RoBERTa model with an untrained classification head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label2idx))

# Enable Multi-GPU Training if available (e.g., Kaggle T4 x2)
if torch.cuda.device_count() > 1:
    print(f"🚀 Detected {torch.cuda.device_count()} GPUs. Activating DataParallel!")
    model = nn.DataParallel(model)

model = model.to(device)

# Standard AdamW Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning Rate Scheduler: Warms up slowly, then follows a cosine decay curve
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

# Initialize Gradient Scaler for Mixed Precision (FP16)
scaler = torch.amp.GradScaler('cuda')

# Inject class weights into the Loss Function
class_weights_tensor = torch.tensor(weights_ordered, dtype=torch.float).to(device)
loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)

In [ ]:
# --- 6. The Core Training Loop (FP16 + Early Stopping) ---
history = {"step_metrics": [], "epoch_metrics": []}
global_step = 0
best_acc = 0
patience = 2  # Number of epochs to wait for improvement before halting
no_improve_epochs = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*40}\n🚀 Epoch {epoch+1}/{EPOCHS}\n{'='*40}")
    model.train()
    
    # ---------------- TRAINING PHASE ----------------
    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Enable Mixed Precision (FP16) for massive speedup without losing accuracy
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            # Force logits to float32 for stable CrossEntropy computation
            loss = loss_fct(logits.float(), labels)
            
        # Backpropagation via Gradient Scaler
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Prevent exploding gradients
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        # Log metrics every 20 steps
        if global_step % 20 == 0:
            preds = torch.argmax(logits, dim=1)
            acc = (preds == labels).float().mean().item()
            history["step_metrics"].append({"step": global_step, "loss": loss.item(), "accuracy": acc})
            
        global_step += 1
        
    # ---------------- VALIDATION PHASE ----------------
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids, attention_mask=attention_mask)
            
            loss = loss_fct(outputs.logits.float(), labels)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
    # Calculate Epoch Metrics
    val_acc = correct / total
    val_loss_avg = val_loss / len(test_loader)
    
    print(f"📊 Validation Loss: {val_loss_avg:.4f} | Validation Accuracy: {val_acc * 100:.2f}%")
    history["epoch_metrics"].append({"epoch": epoch+1, "val_loss": val_loss_avg, "val_acc": val_acc})
    
    # ---------------- EARLY STOPPING LOGIC ----------------
    if val_acc > best_acc:
        best_acc = val_acc
        no_improve_epochs = 0
        
        # Save the model safely (handles DataParallel wrapping)
        model_to_save = model.module if hasattr(model, 'module') else model
        model_to_save.save_pretrained(CHECKPOINT_DIR)
        print("💾 New Best Model Saved!")
    else:
        no_improve_epochs += 1
        print(f"⚠️ No improvement for {no_improve_epochs} epoch(s).")
        
    if no_improve_epochs >= patience:
        print(f"\n🛑 Early Stopping triggered! Validation accuracy plateaued for {patience} consecutive epochs.")
        print(f"🌟 Best Accuracy Secured: {best_acc * 100:.2f}%")
        break

# Export Training History for Local Visualization
with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=4)

In [ ]:
# --- 7. Package Outputs for Download ---
# Compress the 'outputs' folder into a neat ZIP file for local inference and presentation
from IPython.display import FileLink
zip_path = os.path.join(os.getcwd(), "project_outputs.zip")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', OUTPUT_DIR)

print(f"🎉 Training Pipeline Complete! Generating download link...")
display(FileLink('project_outputs.zip'))